In [ ]:
# Install pmdarima package
# !pip install pmdarima

In [ ]:
# Install finpy-tse package
# !pip install finpy-tse

In [ ]:
# Import necessary libraries
import pmdarima as pm
import finpy_tse as fpy
import numpy as np
import pandas as pd
import jdatetime
import warnings
warnings.filterwarnings("ignore")
import time
import winsound

In [ ]:
# Get tickers list from excel file
tickers = pd.read_excel('Symbols.xlsx')

In [ ]:
# Get start and end date
start_date = '1400-01-01'
end_date = str(jdatetime.date.today())

In [ ]:
''' Signal part '''
# -^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-

# Temporary symbol list to buy
sym_list = []

# Temporary symbol DataFrame to buy
buy_list = pd.DataFrame()

''' Main Loop '''
# -^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-

for ticker in tickers.iloc[:]['Ticker']:
    while True:
        try:
            
            ''' Get Data '''
            # -^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-
            
            # Get historical data
            df = fpy.Get_Price_History(stock=ticker, start_date=start_date, end_date=end_date, double_date=True)
            
            # Set index
            df.set_index('Date', inplace=True)

            # Drop unnecessary columns
            df.drop(columns=['Open', 'High', 'Low', 'Final', 'Volume', 'Value', 'No', 'Ticker', 'Name', 'Market'], inplace=True)
            
            ''' ARIMA '''
            # -^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-

            # Log Close Price
            df['Log_Close'] = np.log(df['Close'])

            # Calculation of the log return
            df['Log_Ret'] = df['Log_Close'].diff()

            # Drop First NaN Value
            df.dropna(inplace= True)

            # Create ARIMA model with Log Ret
            model = pm.auto_arima(df['Log_Ret'], trace=False, suppress_warnings=True, seasonal=False, error_action='ignore', stepwise=False)
            
            # Number of the Prediction
            n_pred = 2
            
            # Prediction
            prediction = model.predict(n_periods=n_pred)
            
            # Create the future index
            pred_index_list = []
            pred_index =  df.index[-1] + pd.Timedelta(days=1)
            pred_index_list.append(pred_index)
            
            for i in range(n_pred-1):
                pred_index =  pred_index_list[-1] + pd.Timedelta(days=1)
                pred_index_list.append(pred_index)

            # Create pred dataframe
            pred = pd.DataFrame(index=pred_index_list)

            # Put prediction values into dataframe
            pred['Pred_with_diff'] = np.array(prediction)

            # Last Log_Close value
            last_Log_price = df.iloc[-1]['Log_Close']

            # Undo Differencing
            pred['Pred_Un_diff'] = pred['Pred_with_diff'] + last_Log_price

            # Undo Log
            pred['Pred'] = np.e ** (pred['Pred_Un_diff'])
            
            ''' Prediction '''
            # -^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-

            # Last Price
            Last_Price = df.iloc[-1]['Close']
            
            # Second Last Price
            Second_Last_Price = df.iloc[-2]['Close']

            # First Prediction(ARIMA)
            First_Pred = pred.iloc[-n_pred]['Pred']

            # Second Prediction(ARIMA)
            Second_Pred = pred.iloc[-n_pred+1]['Pred']

            # EMA(20 - 50)
            df['EWMA-20'] = df['Close'].ewm(span=20).mean()
            df['EWMA-50'] = df['Close'].ewm(span=50).mean()
            
            ''' Signal '''
            # -^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-

            # Signal
            con_1_buy = df['EWMA-20'].iloc[-1] > df['EWMA-20'].iloc[-2]
            con_2_buy = df['EWMA-50'].iloc[-1] > df['EWMA-50'].iloc[-2]
            con_3_buy = df['EWMA-20'].iloc[-1] > df['EWMA-50'].iloc[-1]
            con_4_buy = Last_Price > Second_Last_Price
            con_5_buy = Last_Price > df['EWMA-20'].iloc[-1]
            con_6_buy = First_Pred > Last_Price and Second_Pred > First_Pred

            if con_1_buy and con_2_buy and con_3_buy and con_4_buy and con_5_buy and con_6_buy:
                Signal = 'Buy'
            else:
                Signal = 'Sell'

            # Print symbols
            print(20*'-.-')
            # 2: index starts from zero and row one is the column name
            index_num = int(np.where(tickers['Ticker']==ticker)[0]) + 2
            print(f'[{index_num}] Symbol: {ticker}')

            # Print Signal
            if Signal == 'Buy':
                sym_list.append(ticker)       
                print(20*'-^-')
                print(f'[{index_num}] Symbol: {ticker} | Signal: {Signal}')
                print(f'Date: {df.index[-1]} Last Price: {Last_Price}')
                for i in range(n_pred):
                    print(f"Date: {pred.index[i]} Prediction Price: {np.round(pred['Pred'].iloc[i])}")
                print(20*'-^-')
            break
        except:
            pass

''' END LOOP '''
# -^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-^-
             
# Create excel buy list
buy_list['Symbol'] = sym_list
buy_list.to_excel('Buy_List.xlsx', header=False, index=False)

# Make beep sound on Windows
for i in range(3):
    winsound.Beep(frequency=2000, duration=1000)
    time.sleep(1)